# 📊 Data Understanding — IndoToxic2024 Dataset
**Proyek:** Indonesian Hate Speech Analyzer (Kelompok 6)  
**Arsitektur Model:** XLM-RoBERTa  
**Notebook ini** melakukan Exploratory Data Analysis (EDA) pada dataset `indotoxic2024_annotated_data_v2_final.csv`.

---
## Bab 1: Setup & Data Loading

In [ ]:
# === Import Library ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 150)

print("✅ Library berhasil dimuat.")

### 1.1 Memuat Dataset Utama

In [ ]:
import pandas as pd
from pathlib import Path

# Menentukan root directory secara otomatis & dinamis.
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

# Sumber utama tetap dataset project yang diminta.
file_path = project_root / 'data' / 'raw' / 'indotoxic2024_annotated_data_v2_final.csv'

# Fallback hanya untuk portabilitas ketika notebook dipindahkan bersama CSV.
local_fallback = current_dir / 'indotoxic2024_annotated_data_v2_final.csv'

if file_path.exists():
    data_source = file_path
elif local_fallback.exists():
    data_source = local_fallback
else:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Letakkan file pada: "
        f"{file_path} (path utama project) atau {local_fallback}."
    )

df = pd.read_csv(data_source)

print(f"Sumber dataset             : {data_source.resolve()}")
print(f"Jumlah Baris (Data Point)  : {df.shape[0]:,}")
print(f"Jumlah Kolom (Variabel)    : {df.shape[1]}")
print()
df.head()

### 1.2 Memahami Struktur Dataset (Poin 1)
Menampilkan nama kolom, tipe data, dan jumlah data non-null untuk setiap variabel.

In [ ]:
df.info()

In [ ]:
# Daftar nama kolom beserta tipe datanya
print("=== Daftar Kolom & Tipe Data ===")
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f"  {i:2d}. {col:<40s} → {dtype}")

### 1.3 Memeriksa Kualitas Data (Poin 2)
Mengecek data kosong (NaN/missing values), data duplikat, dan distribusi panjang teks.

In [ ]:
# --- Cek Missing Values ---
print("=== Missing Values per Kolom ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah NaN': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Jumlah NaN'] > 0].to_string())
print()

# --- Cek Duplikat ---
n_dup = df.duplicated(subset='text').sum()
print(f"Jumlah teks duplikat: {n_dup}")
print()

# --- Statistik Panjang Teks ---
df['char_count'] = df['text'].astype(str).str.len()
df['word_count'] = df['text'].astype(str).str.split().str.len()

print("=== Statistik Panjang Teks ===")
print(df[['char_count', 'word_count']].describe().round(1).to_string())

In [ ]:
# --- Identifikasi Teks Sangat Pendek & Sangat Panjang ---
very_short = df[df['word_count'] <= 3]
very_long  = df[df['word_count'] >= 200]

print(f"Teks sangat pendek (≤ 3 kata) : {len(very_short):,} baris")
print(f"Teks sangat panjang (≥ 200 kata): {len(very_long):,} baris")
print()

if len(very_short) > 0:
    print("--- Sampel Teks Sangat Pendek ---")
    print(very_short[['text_id', 'text']].head(5).to_string(index=False))

---
## Bab 2: Parsing & Agregasi Label (Majority Voting)

Kolom-kolom label pada dataset masih tersimpan sebagai **string representasi list** (contoh: `"['1', '0']"`).  
Kita perlu mengubahnya menjadi satu label konsensus menggunakan rumus **Majority Voting**:

$$\text{Label} = \begin{cases} 1, & \text{jika rata-rata vote} > 0.5 \\ 0, & \text{jika rata-rata vote} < 0.5 \\ \text{Disagreement}, & \text{jika rata-rata vote} = 0.5 \end{cases}$$

In [ ]:
# === Fungsi Parsing Anotasi List ===

def parse_annotation_list(annotation_str):
    """Mengubah string list anotasi menjadi list angka.
    Contoh: "['1', '0']" -> [1, 0]
    """
    try:
        parsed = ast.literal_eval(annotation_str)
        return [int(x) for x in parsed]
    except (ValueError, SyntaxError):
        return []

print("Fungsi parse_annotation_list() berhasil didefinisikan.")

### 2.1 Analisis Distribusi Jumlah Anotator per Teks
Sebelum melakukan majority voting, kita periksa apakah jumlah annotator di dalam list selalu 2, atau ada teks yang dinilai oleh 1, 3, atau lebih annotator.

In [ ]:
# Menghitung panjang list annotators_id untuk setiap baris
df['num_annotators'] = df['annotators_id'].apply(lambda x: len(parse_annotation_list(x)))

# Tabel frekuensi jumlah annotator
annotator_counts = df['num_annotators'].value_counts().sort_index()
annotator_pct = (annotator_counts / len(df) * 100).round(2)

dist_annotator_df = pd.DataFrame({
    'Jumlah Anotator (Panjang List)': annotator_counts.index,
    'Jumlah Teks (Baris)': annotator_counts.values,
    'Persentase (%)': annotator_pct.values
})

print("=== Distribusi Jumlah Anotator per Teks ===")
print(dist_annotator_df.to_string(index=False))
print()
print(f"Total baris data         : {len(df):,}")
print(f"Rentang jumlah annotator : {df['num_annotators'].min()} s/d {df['num_annotators'].max()} annotator per teks")

In [ ]:
# Visualisasi Distribusi Jumlah Anotator
fig, ax = plt.subplots(figsize=(10, 5))

# Kelompokkan kategori: 1 sampai 6, dan >6
counts_display = annotator_counts[annotator_counts.index <= 6].copy()
other_count = annotator_counts[annotator_counts.index > 6].sum()
if other_count > 0:
    counts_display['>6'] = other_count

bars = ax.bar([str(idx) for idx in counts_display.index], counts_display.values, color='#3498db', edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, counts_display.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Distribusi Jumlah Anotator per Teks (Panjang List)', fontsize=13, fontweight='bold')
ax.set_xlabel('Jumlah Anotator yang Menilai')
ax.set_ylabel('Jumlah Teks')
plt.tight_layout()
plt.show()

### 💡 Interpretasi Jumlah Anotator
Output pada dua cell sebelumnya menunjukkan distribusi jumlah anotator **secara aktual dari dataset**. Karena jumlah anotator dapat berbeda antar-teks, majority voting digunakan untuk membentuk label konsensus. Nilai tepat mengenai jumlah, persentase, dan rentang anotator selalu diambil dari hasil eksekusi notebook, bukan ditulis manual pada Markdown.

In [ ]:
# === Fungsi Majority Voting ===

def majority_vote(annotation_str):
    """Menghitung label konsensus dari string list anotasi.
    Returns:
        1   -> Mayoritas positif (rata-rata > 0.5)
        0   -> Mayoritas negatif (rata-rata < 0.5)
        0.5 -> Disagreement / Tie (rata-rata = 0.5)
    """
    votes = parse_annotation_list(annotation_str)
    if len(votes) == 0:
        return np.nan
    avg = np.mean(votes)
    if avg > 0.5:
        return 1
    elif avg < 0.5:
        return 0
    else:
        return 0.5  # Disagreement

print("Fungsi majority_vote() berhasil didefinisikan.")

In [ ]:
# === Kolom-kolom label yang akan diproses ===
label_columns = [
    'toxicity',
    'identity_attack',
    'threat_incitement_to_violence',
    'insults',
    'profanity_obscenity',
    'sexually_explicit',
    'polarized',
    'related_to_election_2024',
    'is_noise_or_spam_text'
]

# Menerapkan majority voting ke semua kolom label
for col in label_columns:
    new_col = f'{col}_label'
    df[new_col] = df[col].apply(majority_vote)
    print(f"  ✅ {col} → {new_col}")

print()
print("Semua kolom label berhasil diparsing!")
print()

# Menampilkan sampel hasil parsing
sample_cols = ['text_id', 'toxicity', 'toxicity_label', 'insults', 'insults_label']
df[sample_cols].head(5)

---
## Bab 3: Analisis Distribusi Label & Pembuktian Statistik Kritis (Poin 3)

### 3.1 Distribusi Label Toksisitas (Task Utama: Biner)

In [ ]:
# === Distribusi Toxicity (Biner) ===
tox_counts = df['toxicity_label'].value_counts().sort_index()

label_map = {0: 'Non-Toxic (0)', 0.5: 'Disagreement (0.5)', 1: 'Toxic (1)'}
tox_display = tox_counts.rename(index=label_map)

print("=== Distribusi Label Toksisitas ===")
for label, count in tox_display.items():
    pct = count / len(df) * 100
    print(f"  {label:<25s}: {count:>6,} ({pct:.1f}%)")

# Visualisasi Bar Chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(tox_display.index, tox_display.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, tox_display.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Distribusi Label Toksisitas (Majority Voting)', fontsize=14, fontweight='bold')
ax.set_ylabel('Jumlah Data')
ax.set_xlabel('Kategori Label')
plt.tight_layout()
plt.show()

### 3.2 Distribusi Sub-Kategori Multi-Label

In [ ]:
# === Distribusi 5 Sub-Kategori Toksisitas ===
multi_label_cols = [
    'identity_attack_label',
    'threat_incitement_to_violence_label',
    'insults_label',
    'profanity_obscenity_label',
    'sexually_explicit_label'
]

multi_label_names = [
    'Identity Attack\n(SARA)',
    'Threat\n(Ancaman)',
    'Insults\n(Hinaan)',
    'Profanity\n(Kata Kasar)',
    'Sexually\nExplicit'
]

# Menghitung jumlah kelas 1 (positif) untuk setiap sub-kategori
positive_counts = [df[col].apply(lambda x: 1 if x == 1 else 0).sum() for col in multi_label_cols]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(multi_label_names, positive_counts, color='#e74c3c', edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, positive_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Jumlah Teks Positif (Kelas 1) per Sub-Kategori Toksisitas', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Data Berlabel 1')
ax.set_xlabel('Sub-Kategori')
plt.tight_layout()
plt.show()

### 3.3 Rekonstruksi Tabel Statistik 
Memvalidasi angka-angka di **Tabel 5: Distribusi & Statistik Kritis Variabel** pada dokumen `Data_Understanding_Guide.md`.

In [ ]:
# === Rekonstruksi Tabel Statistik Kritis ===
# Memvalidasi angka dari Data_Understanding_Guide.md

all_label_cols = [
    ('toxicity_label',                         'toxicity (Task Utama)'),
    ('polarized_label',                        'polarized'),
    ('identity_attack_label',                  'identity_attack'),
    ('insults_label',                          'insults'),
    ('profanity_obscenity_label',              'profanity_obscenity'),
    ('threat_incitement_to_violence_label',     'threat_incitement_to_violence'),
    ('sexually_explicit_label',                'sexually_explicit'),
    ('is_noise_or_spam_text_label',            'is_noise_or_spam_text'),
    ('related_to_election_2024_label',         'related_to_election_2024'),
]

rows = []
for col, name in all_label_cols:
    total = len(df)
    n_0   = (df[col] == 0).sum()
    n_1   = (df[col] == 1).sum()
    n_dis = (df[col] == 0.5).sum()
    
    # Hitung rasio imbalance (Kelas 0 : Kelas 1)
    ratio = f"1 : {int(round(n_0 / n_1))}" if n_1 > 0 else "N/A"
    
    rows.append({
        'Variabel': name,
        'Kelas 0': f"{n_0:,} ({n_0/total*100:.1f}%)",
        'Kelas 1': f"{n_1:,} ({n_1/total*100:.1f}%)",
        'Disagreement': f"{n_dis:,} ({n_dis/total*100:.1f}%)",
        'Rasio Imbalance': ratio
    })

stats_df = pd.DataFrame(rows)
print("=" * 100)
print("TABEL STATISTIK ")
print("=" * 100)
print(stats_df.to_string(index=False))

---
## Bab 4: Analisis Karakteristik Teks & Pembuktian Tantangan Data (Poin 4 & 6)

### 4.1 Distribusi Panjang Teks (Karakter & Kata)

In [ ]:
# === Histogram Panjang Teks: Toxic vs Non-Toxic ===
df_toxic     = df[df['toxicity_label'] == 1]
df_nontoxic  = df[df['toxicity_label'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram Jumlah Kata
axes[0].hist(df_nontoxic['word_count'], bins=50, alpha=0.6, label='Non-Toxic', color='#2ecc71', edgecolor='black', linewidth=0.3)
axes[0].hist(df_toxic['word_count'],    bins=50, alpha=0.6, label='Toxic',     color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[0].set_title('Distribusi Jumlah Kata', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Histogram Jumlah Karakter
axes[1].hist(df_nontoxic['char_count'], bins=50, alpha=0.6, label='Non-Toxic', color='#2ecc71', edgecolor='black', linewidth=0.3)
axes[1].hist(df_toxic['char_count'],    bins=50, alpha=0.6, label='Toxic',     color='#e74c3c', edgecolor='black', linewidth=0.3)
axes[1].set_title('Distribusi Jumlah Karakter', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Karakter')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.suptitle('Perbandingan Panjang Teks: Toxic vs Non-Toxic', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistik rata-rata
print("=== Rata-rata Panjang Teks ===")
print(f"  Non-Toxic → Kata: {df_nontoxic['word_count'].mean():.1f}, Karakter: {df_nontoxic['char_count'].mean():.1f}")
print(f"  Toxic     → Kata: {df_toxic['word_count'].mean():.1f}, Karakter: {df_toxic['char_count'].mean():.1f}")

### 4.2 Kata yang Paling Sering Muncul pada Teks Toxic

In [ ]:
# === Top 20 Kata Paling Sering Muncul di Teks Toxic ===
from collections import Counter

# Mengambil semua kata dari teks berlabel Toxic
all_words_toxic = ' '.join(df_toxic['text'].astype(str).str.lower()).split()

# Menghitung frekuensi kemunculan
word_freq = Counter(all_words_toxic)
top_20 = word_freq.most_common(20)

# Visualisasi Horizontal Bar Chart
words, counts = zip(*top_20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(words)), counts, color='#e74c3c', edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_title('Top 20 Kata Paling Sering Muncul di Teks Toxic', fontsize=13, fontweight='bold')
ax.set_xlabel('Frekuensi Kemunculan')
plt.tight_layout()
plt.show()

### 4.3 Word Cloud — Teks Toxic

In [ ]:
%pip install wordcloud

In [ ]:
# === Word Cloud: Kata Dominan pada Teks Toxic (Diperbarui) ===
import re

try:
    from wordcloud import WordCloud

    # 1. Menggabungkan semua teks menjadi huruf kecil
    text_toxic_all = ' '.join(df_toxic['text'].astype(str).str.lower())
    
    # 2. Pembersihan ringan: Hapus URL dan @mention agar Word Cloud bersih
    text_toxic_all = re.sub(r'http\S+|www\S+|https\S+', '', text_toxic_all, flags=re.MULTILINE)
    text_toxic_all = re.sub(r'\@\w+', '', text_toxic_all)
    
    # 3. Mendefinisikan kata-kata umum yang tidak penting (Stopwords Bahasa Indonesia & Slang)
    indo_stopwords = set([
        'yang', 'di', 'dan', 'ini', 'itu', 'untuk', 'dengan', 'dari', 'ke', 'pada', 
        'dalam', 'adalah', 'juga', 'ada', 'orang', 'yg', 'ya', 'aja', 
        'kalo', 'buat', 'sama', 'bisa', 'karena', 'kalau', 'akan', 'aku', 'saya',
        'dia', 'mereka', 'kita', 'kamu', 'udah', 'gak', 'nya', 'kok', 'sih', 'lagi',
        'lebih', 'banyak', 'sudah', 'baru', 'jadi'
    ])

    # 4. Generate Word Cloud dengan memasukkan parameter stopwords
    wordcloud = WordCloud(
        width=1000, height=500,
        background_color='white',
        colormap='Reds',      # Warna merah cocok untuk konteks bahaya/toxic
        max_words=100,        # Kurangi jadi 100 agar kata besar lebih jelas terbaca
        stopwords=indo_stopwords, # <--- KUNCI UTAMANYA DI SINI
        collocations=False
    ).generate(text_toxic_all)

    # 5. Visualisasi
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — Kata Dominan pada Teks Berlabel Toxic', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠️ Library 'wordcloud' belum terinstal.")
    print("   Jalankan: pip install wordcloud")

In [ ]:
import re
import matplotlib.pyplot as plt

# 1. Pengecekan dan instalasi otomatis untuk NLTK
try:
    import nltk
    from nltk.corpus import stopwords
except ImportError:
    print("Modul 'nltk' belum terinstal. Memulai instalasi...")
    %pip install nltk
    import nltk
    from nltk.corpus import stopwords

# 2. Pengecekan dan instalasi otomatis untuk WordCloud
try:
    from wordcloud import WordCloud
except ImportError:
    print("Modul 'wordcloud' belum terinstal. Memulai instalasi...")
    %pip install wordcloud
    from wordcloud import WordCloud


# --- PROSES PEMBUATAN WORDCLOUD ---

# Download database stopwords dari NLTK (Hanya akan di-download jika belum ada)
nltk.download('stopwords', quiet=True)

# Ambil list stopwords resmi bahasa Indonesia dari NLTK
nltk_stopwords = set(stopwords.words('indonesian'))

# TAMBAHKAN slang/kata gaul secara manual karena NLTK tidak tahu kata gaul
slang_stopwords = {'yg', 'aja', 'kalo', 'buat', 'sama', 'udah', 'gak', 'nya', 'kok', 'sih', 'nih', 'tuh'}

# Gabungkan keduanya
negation_keep = {'tidak', 'bukan', 'jangan', 'belum', 'tanpa', 'tak', 'nggak', 'gak'}
final_stopwords = nltk_stopwords.union(slang_stopwords) - negation_keep

# Preprocessing teks dasar
text_toxic_all = ' '.join(df_toxic['text'].astype(str).str.lower())
text_toxic_all = re.sub(r'http\S+|www\S+|https\S+', '', text_toxic_all)
text_toxic_all = re.sub(r'\@\w+', '', text_toxic_all)

# Masukkan final_stopwords ke dalam WordCloud
wordcloud = WordCloud(
    width=1000, height=500,
    background_color='white',
    colormap='Reds',
    max_words=100,
    stopwords=final_stopwords, # Gunakan gabungan NLTK + Manual
    collocations=False
).generate(text_toxic_all)

# Visualisasi
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
plt.tight_layout()
plt.show()


### 4.4 Heatmap Korelasi Antar Sub-Label
Melihat apakah kalimat hinaan (*insults*) sering muncul bersamaan dengan kata kasar (*profanity*), sebagai justifikasi pemilihan fungsi aktivasi **Sigmoid** (independen per label).

In [ ]:
# === Heatmap Korelasi Antar Sub-Label ===
corr_cols = [
    'toxicity_label',
    'identity_attack_label',
    'threat_incitement_to_violence_label',
    'insults_label',
    'profanity_obscenity_label',
    'sexually_explicit_label'
]

corr_names = ['Toxicity', 'SARA', 'Ancaman', 'Hinaan', 'Kata Kasar', 'Seksual']

# Hanya ambil data yang bukan disagreement (0 atau 1)
df_corr = df[corr_cols].copy()
df_corr = df_corr[(df_corr != 0.5).all(axis=1)]

correlation = df_corr.corr()
correlation.index = corr_names
correlation.columns = corr_names

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=-0.1, vmax=1, linewidths=0.5, ax=ax,
            square=True)
ax.set_title('Heatmap Korelasi Antar Label Toksisitas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bab 5: EDA Text Mining Lengkap
Bagian berikut melengkapi analisis sebelumnya. Seluruh statistik, tabel, rasio, contoh teks, dan rekomendasi dihitung **langsung dari `df`** agar tidak bergantung pada angka yang ditulis manual.

### 5.1 Pemeriksaan Struktur Dataset Lengkap
Melengkapi pemeriksaan awal dengan `shape`, daftar kolom, tipe data, `info()`, data awal/akhir, deskripsi, serta identifikasi kolom teks dan label berdasarkan struktur dataset yang benar-benar terbaca.

In [ ]:
from io import StringIO

print("=== SHAPE DATASET ===")
print(f"Baris: {df.shape[0]:,} | Kolom: {df.shape[1]}")
print("\n=== NAMA SELURUH KOLOM ===")
print(list(df.columns))
print("\n=== TIPE DATA ===")
print(df.dtypes.to_string())
print("\n=== INFO DATASET ===")
buffer = StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

print("=== 5 DATA PERTAMA ===")
display(df.head())
print("=== 5 DATA TERAKHIR ===")
display(df.tail())

text_col = 'text' if 'text' in df.columns else next((c for c in df.select_dtypes(include='object').columns if df[c].astype(str).str.len().mean() > 20), None)
label_cols_detected = [c for c in df.columns if c.endswith('_label')]
main_label_col = 'toxicity_label' if 'toxicity_label' in df.columns else (label_cols_detected[0] if label_cols_detected else None)

print("\n=== IDENTIFIKASI KOLOM ===")
print(f"Kolom teks utama : {text_col}")
print(f"Target utama     : {main_label_col}")
print(f"Kolom label hasil agregasi ({len(label_cols_detected)}): {label_cols_detected}")

print("\n=== DESKRIPSI DATA RELEVAN ===")
display(df.describe(include='all').T)

print("\nINTERPRETASI:")
print(f"Dataset yang terbaca memiliki {df.shape[0]:,} dokumen dan {df.shape[1]} kolom. "
      f"Kolom teks yang digunakan untuk EDA adalah '{text_col}', sedangkan target utama adalah '{main_label_col}'.")

### 5.2 Missing Value, None, String Kosong, Whitespace, dan Teks Tidak Valid
Tidak hanya `NaN`, bagian ini memeriksa bentuk kekosongan lain pada setiap kolom dan secara khusus mengevaluasi validitas kolom teks.

In [ ]:
# Tabel missing per kolom
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(4)
missing_table = pd.DataFrame({'Kolom': df.columns, 'Missing': missing_count.values, 'Persentase': missing_pct.values})
display(missing_table)

text_series = df[text_col]
text_as_str = text_series.fillna('').astype(str)
is_blank = text_as_str.eq('')
is_whitespace = text_as_str.str.fullmatch(r'\s+', na=False)
is_symbol_only = text_as_str.str.fullmatch(r'[^\w\s]+', na=False)
is_none_literal = text_as_str.str.strip().str.lower().isin({'none', 'null', 'nan'})
invalid_text = text_series.isna() | is_blank | is_whitespace | is_symbol_only | is_none_literal

text_validity = pd.DataFrame({
    'Jenis': ['NaN/None aktual', 'String kosong', 'Whitespace saja', 'Simbol saja', 'Literal none/null/nan', 'Total teks tidak valid (union)'],
    'Jumlah': [text_series.isna().sum(), is_blank.sum(), is_whitespace.sum(), is_symbol_only.sum(), is_none_literal.sum(), invalid_text.sum()]
})
text_validity['Persentase'] = (text_validity['Jumlah'] / len(df) * 100).round(4)
display(text_validity)

print("INTERPRETASI:")
if invalid_text.sum() == 0:
    print("Tidak ditemukan teks kosong/tidak valid menurut pemeriksaan NaN, empty string, whitespace, simbol-saja, dan literal null. Tidak diperlukan pembuangan teks kosong sebelum pemodelan.")
else:
    print(f"Ditemukan {invalid_text.sum():,} teks tidak valid ({invalid_text.mean()*100:.3f}%). Baris ini perlu ditangani sebelum tokenisasi/pemodelan karena tidak membawa isi linguistik yang memadai.")

### 5.3 Data Duplikat, Conflicting Labels, dan Potensi Leakage
Duplikat tidak langsung dihapus. Analisis membedakan duplikat seluruh baris, duplikat teks, konflik label pada teks identik, dan duplikasi setelah normalisasi ringan sebagai indikator *near-duplicate*.

In [ ]:
import re

n_dup_rows = int(df.duplicated().sum())
n_dup_text = int(df.duplicated(subset=[text_col], keep=False).sum())
n_dup_text_excess = int(df.duplicated(subset=[text_col]).sum())

def normalize_for_dup(x):
    x = str(x).lower()
    x = re.sub(r'https?://\S+|www\.\S+', ' URL ', x)
    x = re.sub(r'@\w+', ' USER ', x)
    x = re.sub(r'\s+', ' ', x)
    x = re.sub(r'[^\w\s]', '', x)
    return x.strip()

eda_norm_text = df[text_col].fillna('').map(normalize_for_dup)
n_norm_dup_excess = int(eda_norm_text.duplicated().sum())

dup_summary = pd.DataFrame({
    'Metrik': ['Duplicate rows identik', 'Baris yang termasuk grup teks duplikat', 'Duplikat teks berlebih', 'Duplikat setelah normalisasi ringan (indikasi near-duplicate)'],
    'Jumlah': [n_dup_rows, n_dup_text, n_dup_text_excess, n_norm_dup_excess]
})
dup_summary['Persentase'] = (dup_summary['Jumlah']/len(df)*100).round(3)
display(dup_summary)

if n_dup_text > 0:
    print("Contoh teks duplikat:")
    display(df[df.duplicated(subset=[text_col], keep=False)][[c for c in ['text_id', text_col, main_label_col] if c in df.columns]].sort_values(text_col).head(12))

conflict_examples = pd.DataFrame()
conflict_text_count = 0
if main_label_col:
    tmp_conf = df[[text_col, main_label_col]].dropna()
    conflict_keys = tmp_conf.groupby(text_col)[main_label_col].nunique()
    conflict_keys = conflict_keys[conflict_keys > 1].index
    conflict_text_count = len(conflict_keys)
    if conflict_text_count:
        conflict_examples = df[df[text_col].isin(conflict_keys)][[c for c in ['text_id', text_col, main_label_col] if c in df.columns]].sort_values(text_col)
        print(f"Teks identik dengan conflicting labels pada target utama: {conflict_text_count:,} teks unik")
        display(conflict_examples.head(12))
    else:
        print("Tidak ditemukan teks identik dengan label target utama yang berbeda.")

print("\nREKOMENDASI:")
print("Duplikat dapat menaikkan bobot frekuensi secara tidak proporsional, mencerminkan spam, dan menimbulkan data leakage bila salinan/near-duplicate terpisah antara train dan test. Evaluasi dan deduplikasi sebaiknya dilakukan sebelum train-test split, sambil meninjau kasus conflicting labels secara manual atau dengan aturan konsensus yang konsisten.")

### 5.4 Distribusi Dokumen dan Label Target
Distribusi dihitung dari label hasil majority voting. Nilai `0.5` tetap ditampilkan sebagai *disagreement* dan tidak diam-diam digabungkan ke kelas lain.

In [ ]:
label_distribution = None
if main_label_col:
    counts = df[main_label_col].value_counts(dropna=False).sort_index()
    label_distribution = pd.DataFrame({'Label': counts.index.astype(str), 'Jumlah': counts.values})
    label_distribution['Persentase'] = (label_distribution['Jumlah']/len(df)*100).round(3)
    display(label_distribution)

    fig, ax = plt.subplots(figsize=(8,5))
    ax.bar(label_distribution['Label'], label_distribution['Jumlah'])
    ax.set_title(f'Distribusi Kelas — {main_label_col}')
    ax.set_xlabel('Label')
    ax.set_ylabel('Jumlah Dokumen')
    plt.tight_layout(); plt.show()

    binary_counts = df.loc[df[main_label_col].isin([0,1]), main_label_col].value_counts()
    if len(binary_counts) >= 2:
        maj = int(binary_counts.max()); mino = int(binary_counts.min())
        imbalance_ratio = maj / mino if mino else np.inf
        if imbalance_ratio < 1.5:
            balance_status = 'relatif seimbang'
        elif imbalance_ratio < 3:
            balance_status = 'sedikit tidak seimbang'
        else:
            balance_status = 'class imbalance yang cukup besar'
        print(f"Rasio kelas mayoritas : minoritas = {imbalance_ratio:.2f} : 1 → {balance_status}.")
        print("Implikasi: jangan hanya mengandalkan accuracy. Gunakan precision, recall, F1-score, macro F1, dan confusion matrix; pertimbangkan class_weight/strategi imbalance bila rasio memang signifikan.")

### 5.5 Analisis Panjang Teks
Fitur EDA bersifat sementara dan tidak mengubah teks asli. Jumlah kalimat dihitung secara heuristik dari tanda `.`, `!`, dan `?`.

In [ ]:
eda_text = df[text_col].fillna('').astype(str)
eda_char_count = eda_text.str.len()
eda_word_count = eda_text.str.findall(r'\b\w+\b', flags=re.UNICODE).str.len()
eda_sentence_count = eda_text.str.split(r'[.!?]+', regex=True).apply(lambda parts: len([p for p in parts if str(p).strip()]))

length_stats = pd.DataFrame({
    'Karakter': eda_char_count.describe(percentiles=[.25,.5,.75]),
    'Kata': eda_word_count.describe(percentiles=[.25,.5,.75]),
    'Kalimat_heuristik': eda_sentence_count.describe(percentiles=[.25,.5,.75])
}).loc[['mean','50%','min','max','25%','75%','std']].rename(index={'50%':'median','25%':'Q1','75%':'Q3'})
display(length_stats.round(2))

fig, ax = plt.subplots(figsize=(10,5)); ax.hist(eda_char_count, bins=60); ax.set_title('Histogram Panjang Teks berdasarkan Karakter'); ax.set_xlabel('Jumlah Karakter'); ax.set_ylabel('Jumlah Dokumen'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(10,5)); ax.hist(eda_word_count, bins=60); ax.set_title('Histogram Panjang Teks berdasarkan Jumlah Kata'); ax.set_xlabel('Jumlah Kata'); ax.set_ylabel('Jumlah Dokumen'); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(10,4)); ax.boxplot(eda_word_count, vert=False); ax.set_title('Boxplot Panjang Teks berdasarkan Jumlah Kata'); ax.set_xlabel('Jumlah Kata'); plt.tight_layout(); plt.show()

q1, q3 = eda_word_count.quantile([.25,.75]); iqr=q3-q1
upper_outlier = q3 + 1.5*iqr
very_short_mask = eda_word_count <= 3
very_long_mask = eda_word_count > upper_outlier
print(f"Teks sangat pendek (≤3 kata): {very_short_mask.sum():,} ({very_short_mask.mean()*100:.2f}%)")
print(f"Outlier panjang (> Q3 + 1.5×IQR = {upper_outlier:.1f} kata): {very_long_mask.sum():,} ({very_long_mask.mean()*100:.2f}%)")

if main_label_col:
    compare_len = pd.DataFrame({'label':df[main_label_col], 'char_count':eda_char_count, 'word_count':eda_word_count}).groupby('label').agg(['count','mean','median'])
    display(compare_len.round(2))

p95_words = eda_word_count.quantile(.95)
p99_words = eda_word_count.quantile(.99)
print(f"Persentil 95 jumlah kata: {p95_words:.0f}; persentil 99: {p99_words:.0f}.")
print("Implikasi Transformer: nilai max_length sebaiknya diputuskan setelah melihat distribusi token dari tokenizer model. Statistik kata ini memberi indikasi awal agar truncation tidak terlalu agresif dan padding tidak berlebihan.")

### 5.6 Karakter Khusus dan Elemen Digital
Untuk hate speech, elemen emosional tidak otomatis dihapus. Analisis ini mengukur prevalensinya terlebih dahulu.

In [ ]:
patterns = {
    'URL': r'https?://\S+|www\.\S+',
    'Mention @username': r'@\w+',
    'Hashtag': r'#\w+',
    'Angka': r'\d',
    'Emoji (heuristik Unicode)': r'[\U0001F300-\U0001FAFF\u2600-\u27BF]',
    'Tanda baca berlebihan': r'([!?.,])\1{2,}',
    'Huruf kapital berlebihan': r'\b[A-Z]{4,}\b',
    'Newline': r'\n|\r',
    'Karakter non-alfanumerik': r'[^\w\s]'
}
rows=[]
for name, pat in patterns.items():
    mask=eda_text.str.contains(pat, regex=True, na=False)
    rows.append({'Elemen':name,'Jumlah Dokumen':int(mask.sum()),'Persentase':round(mask.mean()*100,3)})
element_table=pd.DataFrame(rows)
display(element_table)

recommend_map = {
    'URL':'ganti token khusus <URL> bila cukup sering; isi URL biasanya tidak perlu dipertahankan',
    'Mention @username':'ganti token <USER> untuk mengurangi sparsity tanpa kehilangan sinyal adanya mention',
    'Hashtag':'pertahankan isi hashtag atau pisahkan tanda #; dapat membawa topik/stance penting',
    'Angka':'pertahankan atau normalisasi <NUM> sesuai konteks; jangan hapus otomatis',
    'Emoji (heuristik Unicode)':'pertahankan atau konversi ke deskripsi/token emoji karena bisa membawa emosi/sarkasme',
    'Tanda baca berlebihan':'normalisasi jumlahnya tetapi pertahankan sinyal intensitas',
    'Huruf kapital berlebihan':'jika lowercase dilakukan, pertimbangkan fitur/token penanda ALLCAPS agar intensitas tidak hilang',
    'Newline':'normalisasi ke spasi bila tidak bermakna struktural',
    'Karakter non-alfanumerik':'bersihkan selektif; jangan menghapus emoji/hashtag/punctuation emosional secara membabi buta'
}
rec_df=element_table.copy(); rec_df['Rekomendasi']=rec_df['Elemen'].map(recommend_map)
display(rec_df)

### 5.7 Bahasa, Slang, Singkatan, Variasi Penulisan, dan Typo
Bagian ini bersifat eksploratif. Kandidat ditampilkan hanya bila benar-benar ditemukan di corpus; tidak ada normalisasi permanen.

In [ ]:
from collections import Counter

basic_tokens = re.findall(r"(?u)\b[\w']+\b", ' '.join(eda_text.str.lower()))
basic_counter = Counter(basic_tokens)

slang_map = {
    'gak':'tidak','ga':'tidak','nggak':'tidak','tdk':'tidak','tak':'tidak',
    'yg':'yang','dgn':'dengan','dr':'dari','krn':'karena','karna':'karena','kalo':'kalau','kl':'kalau',
    'udh':'sudah','udah':'sudah','blm':'belum','bgt':'banget','banget':'sangat','aja':'saja','sm':'sama',
    'gw':'saya/aku','gue':'saya/aku','lu':'kamu','lo':'kamu','org':'orang','jd':'jadi','jgn':'jangan'
}
slang_rows=[{'Bentuk Asli':w,'Frekuensi':basic_counter[w],'Dugaan Normalisasi':norm} for w,norm in slang_map.items() if basic_counter[w]>0]
slang_df=pd.DataFrame(sorted(slang_rows,key=lambda x:x['Frekuensi'],reverse=True))
print("Kandidat slang/singkatan yang benar-benar ditemukan:")
display(slang_df.head(30) if len(slang_df) else pd.DataFrame(columns=['Bentuk Asli','Frekuensi','Dugaan Normalisasi']))

elong_counter=Counter()
for tok,c in basic_counter.items():
    if re.search(r'(.)\1{2,}', tok): elong_counter[tok]=c
elong_df=pd.DataFrame(elong_counter.most_common(20), columns=['Bentuk Memanjang/Variasi','Frekuensi'])
print("Kandidat kata memanjang/typo berbasis repeated-character:")
display(elong_df)

english_markers={'the','and','is','are','you','your','this','that','of','to','for','with','not','fuck','shit','stupid'}
found_en=[(w,basic_counter[w]) for w in english_markers if basic_counter[w]>0]
print("Marker bahasa Inggris yang ditemukan (indikasi code-mixing, bukan deteksi bahasa formal):")
display(pd.DataFrame(sorted(found_en,key=lambda x:x[1],reverse=True), columns=['Token','Frekuensi']))

if len(slang_df):
    ex_words=set(slang_df['Bentuk Asli'].head(10))
    mask=eda_text.str.lower().apply(lambda s:any(re.search(rf'\b{re.escape(w)}\b',s) for w in ex_words))
    print("Contoh nyata teks yang mengandung kandidat slang/singkatan:")
    display(df.loc[mask,[c for c in ['text_id',text_col,main_label_col] if c in df.columns]].head(8))
print("Catatan: kandidat typo tidak dapat dipastikan hanya dari statistik corpus. Normalisasi perlu kamus/aturan yang divalidasi agar kata ofensif, nama, dan slang penting tidak berubah makna.")

### 5.8 Word Frequency Sebelum dan Sesudah Stopword Removal
Stopword Bahasa Indonesia digunakan secara konservatif. Negasi seperti `tidak`, `bukan`, `jangan`, `belum`, `tanpa`, `tak`, `gak`, dan `nggak` sengaja dipertahankan.

In [ ]:
# Stopword konservatif: kata fungsi umum, tetapi negasi dipertahankan.
stopwords_id = {
    'yang','dan','di','ke','dari','dengan','untuk','pada','dalam','adalah','itu','ini','ada','juga','atau','oleh','sebagai','karena','jadi','sudah','akan','saya','aku','kamu','dia','mereka','kita','kami','nya','pun','lah','kah','para','sebuah','seorang','tersebut','saat','ketika','agar','supaya','hingga','sampai','lebih','sangat','bisa','dapat','buat','sama','aja','ya','yg','kok','sih','nih','tuh'
}
negation_keep={'tidak','bukan','jangan','belum','tanpa','tak','gak','nggak'}
stopwords_id -= negation_keep

clean_for_tokens = eda_text.str.lower().str.replace(r'https?://\S+|www\.\S+',' ',regex=True).str.replace(r'@\w+',' ',regex=True)
all_tokens = re.findall(r'(?u)\b\w+\b', ' '.join(clean_for_tokens))
all_counter=Counter(all_tokens)
filtered_tokens=[t for t in all_tokens if t not in stopwords_id and len(t)>1]
filtered_counter=Counter(filtered_tokens)

def top_table(counter,n=20): return pd.DataFrame(counter.most_common(n), columns=['Kata','Frekuensi'])
top_before=top_table(all_counter); top_after=top_table(filtered_counter)
print("Top 20 sebelum stopword removal:"); display(top_before)
print("Top 20 setelah stopword removal:"); display(top_after)

for title,tab in [('Top 20 Kata — Sebelum Stopword Removal',top_before),('Top 20 Kata — Setelah Stopword Removal',top_after)]:
    fig,ax=plt.subplots(figsize=(10,6)); ax.barh(tab['Kata'][::-1],tab['Frekuensi'][::-1]); ax.set_title(title); ax.set_xlabel('Frekuensi'); ax.set_ylabel('Kata'); plt.tight_layout(); plt.show()

### 5.9 Stopword Analysis
Analisis ini mengukur seberapa dominan stopword, bukan langsung menghapusnya dari dataset.

In [ ]:
stopword_counter=Counter(t for t in all_tokens if t in stopwords_id)
stopword_total=sum(stopword_counter.values())
prop_stop=stopword_total/len(all_tokens)*100 if all_tokens else 0
stop_top=pd.DataFrame(stopword_counter.most_common(20),columns=['Stopword','Frekuensi'])
display(stop_top)
print(f"Total token: {len(all_tokens):,}")
print(f"Token yang termasuk stopword konservatif: {stopword_total:,} ({prop_stop:.2f}%)")
print("Perubahan kata dominan dapat dibandingkan dari tabel sebelum/sesudah pada bagian 5.8.")
print(f"Negasi yang sengaja dipertahankan dan ditemukan: {[(w, all_counter[w]) for w in sorted(negation_keep) if all_counter[w]>0]}")
print("Rekomendasi: hapus stopword fungsi yang sangat umum hanya bila model/representasi membutuhkan; jangan menghapus negasi karena dapat membalik polaritas dan konteks hate speech.")

### 5.10 Vocabulary Size dan Lexical Diversity

In [ ]:
total_tokens=len(all_tokens); unique_tokens=len(all_counter)
once=sum(1 for c in all_counter.values() if c==1); twice=sum(1 for c in all_counter.values() if c==2)
lex_div=unique_tokens/total_tokens if total_tokens else np.nan
vocab_stats=pd.DataFrame({
    'Metrik':['Total token','Unique token / vocabulary size','Lexical diversity','Token muncul 1 kali','Token muncul 2 kali'],
    'Nilai':[total_tokens,unique_tokens,round(lex_div,4),once,twice]
})
display(vocab_stats)
print(f"Vocabulary per dokumen = {unique_tokens/max(len(df),1):.3f} unique token/dokumen (indikator kasar).")
if total_tokens:
    print(f"Hapax legomena (muncul sekali) mencakup {once/unique_tokens*100:.2f}% dari vocabulary." if unique_tokens else '')

### 5.11 Rare Words Analysis
Rare words tidak otomatis dihapus karena bisa memuat kata ofensif spesifik, nama target, slang, atau bentuk khas komunitas.

In [ ]:
rare1=[w for w,c in all_counter.items() if c==1]
rare2=[w for w,c in all_counter.items() if c==2]
rare5=[w for w,c in all_counter.items() if c<=5]
rare_stats=pd.DataFrame({'Frekuensi':['= 1','= 2','≤ 5'],'Jumlah token unik':[len(rare1),len(rare2),len(rare5)]})
display(rare_stats)
print("Contoh token frekuensi 1:", rare1[:30])
print("Contoh token frekuensi 2:", rare2[:30])

rare_examples=[]
for w in rare5[:500]:
    category=[]
    if re.search(r'\d',w): category.append('kode/angka')
    if re.search(r'(.)\1{2,}',w): category.append('typo/elongasi')
    if w in slang_map: category.append('slang/singkatan')
    if len(w)>=15: category.append('token sangat panjang/nama/URL residual')
    if category: rare_examples.append((w,all_counter[w],', '.join(category)))
rare_class_df=pd.DataFrame(rare_examples[:30],columns=['Rare Word','Frekuensi','Kemungkinan Asal (heuristik)'])
display(rare_class_df)
print("Interpretasi: heuristik hanya membantu penyaringan kandidat. Rare words harus ditinjau sebelum dibuang karena kata target kelompok atau makian tertentu justru dapat jarang tetapi sangat informatif.")

### 5.12 N-Gram Analysis: Unigram, Bigram, Trigram
N-gram dihitung dari teks dengan pembersihan ringan dan stopword konservatif. Ini membantu melihat frasa yang mungkin lebih informatif daripada token tunggal.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def lightly_clean(s):
    s=str(s).lower(); s=re.sub(r'https?://\S+|www\.\S+',' URL ',s); s=re.sub(r'@\w+',' USER ',s); s=re.sub(r'\s+',' ',s); return s.strip()
eda_clean_text=eda_text.map(lightly_clean)

def top_ngrams(texts, ngram_range, top_n=20, min_df=2):
    vec=CountVectorizer(ngram_range=ngram_range, stop_words=list(stopwords_id), token_pattern=r'(?u)\b\w+\b', min_df=min_df, max_features=50000)
    X=vec.fit_transform(texts)
    sums=np.asarray(X.sum(axis=0)).ravel(); terms=np.array(vec.get_feature_names_out())
    idx=np.argsort(sums)[-top_n:][::-1]
    return pd.DataFrame({'N-Gram':terms[idx],'Frekuensi':sums[idx].astype(int)})

uni_top=top_ngrams(eda_clean_text,(1,1)); bi_top=top_ngrams(eda_clean_text,(2,2)); tri_top=top_ngrams(eda_clean_text,(3,3))
for title,tab in [('Top 20 Unigram',uni_top),('Top 20 Bigram',bi_top),('Top 20 Trigram',tri_top)]:
    print(title); display(tab)
    fig,ax=plt.subplots(figsize=(10,6)); ax.barh(tab['N-Gram'][::-1],tab['Frekuensi'][::-1]); ax.set_title(title); ax.set_xlabel('Frekuensi'); ax.set_ylabel('N-Gram'); plt.tight_layout(); plt.show()

if main_label_col:
    per_class_bigrams={}
    for lab in sorted(df[main_label_col].dropna().unique()):
        texts=eda_clean_text[df[main_label_col].eq(lab)]
        if len(texts)>=2:
            try: per_class_bigrams[str(lab)]=top_ngrams(texts,(2,2),10,min_df=1)
            except ValueError: pass
    for lab,tab in per_class_bigrams.items():
        print(f"Top bigram label {lab}:"); display(tab)
print("Interpretasi: bigram/trigram yang konsisten per kelas dapat menangkap konteks yang hilang pada unigram, termasuk negasi atau target ujaran.")

### 5.13 TF-IDF Analysis
TF-IDF menekankan term yang relatif khas pada dokumen/corpus, berbeda dari frekuensi mentah yang hanya menghitung kemunculan.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec=TfidfVectorizer(stop_words=list(stopwords_id), token_pattern=r'(?u)\b\w+\b', min_df=2, max_df=0.98, max_features=30000, sublinear_tf=True)
X_tfidf=tfidf_vec.fit_transform(eda_clean_text)
terms=np.array(tfidf_vec.get_feature_names_out())
mean_weights=np.asarray(X_tfidf.mean(axis=0)).ravel()
idx=np.argsort(mean_weights)[-20:][::-1]
top_tfidf=pd.DataFrame({'Term':terms[idx],'Mean TF-IDF':mean_weights[idx]})
print(f"Ukuran matriks TF-IDF: {X_tfidf.shape}")
print(f"Jumlah fitur: {len(terms):,}")
display(top_tfidf)
fig,ax=plt.subplots(figsize=(10,6)); ax.barh(top_tfidf['Term'][::-1],top_tfidf['Mean TF-IDF'][::-1]); ax.set_title('Top 20 Term berdasarkan Mean TF-IDF'); ax.set_xlabel('Mean TF-IDF'); ax.set_ylabel('Term'); plt.tight_layout(); plt.show()

per_class_tfidf={}
if main_label_col:
    labels_sorted=sorted(df[main_label_col].dropna().unique())
    for lab in labels_sorted:
        mask=df[main_label_col].eq(lab).to_numpy()
        if mask.sum():
            cw=np.asarray(X_tfidf[mask].mean(axis=0)).ravel(); ci=np.argsort(cw)[-15:][::-1]
            per_class_tfidf[str(lab)]=pd.DataFrame({'Term':terms[ci],'Mean TF-IDF':cw[ci]})
            print(f"Top TF-IDF label {lab}:"); display(per_class_tfidf[str(lab)])
print("Perbedaan: word frequency menunjukkan kata yang paling sering; TF-IDF menurunkan pengaruh term yang terlalu umum dan menaikkan term yang lebih khas. Pada tahap ini keduanya hanya digunakan sebagai EDA.")

### 5.14 Word Cloud Seluruh Corpus dan Per Kelas
Word cloud hanya visualisasi pendukung. Pembersihan ringan menghilangkan URL/mention sebagai string mentah, tetapi negasi dipertahankan.

In [ ]:
from wordcloud import WordCloud

def wc_text(texts):
    joined=' '.join(map(str,texts)).lower(); joined=re.sub(r'https?://\S+|www\.\S+',' ',joined); joined=re.sub(r'@\w+',' ',joined); return joined

wc_all=WordCloud(width=1000,height=450,background_color='white',max_words=120,stopwords=stopwords_id,collocations=False,random_state=42).generate(wc_text(eda_text))
fig,ax=plt.subplots(figsize=(14,6)); ax.imshow(wc_all,interpolation='bilinear'); ax.axis('off'); ax.set_title('Word Cloud — Seluruh Corpus'); plt.tight_layout(); plt.show()

if main_label_col:
    for lab in sorted(df[main_label_col].dropna().unique()):
        subset=eda_text[df[main_label_col].eq(lab)]
        if len(subset)==0: continue
        wct=wc_text(subset)
        if not wct.strip(): continue
        wc=WordCloud(width=1000,height=400,background_color='white',max_words=100,stopwords=stopwords_id,collocations=False,random_state=42).generate(wct)
        fig,ax=plt.subplots(figsize=(14,5)); ax.imshow(wc,interpolation='bilinear'); ax.axis('off'); ax.set_title(f'Word Cloud — Label {lab}'); plt.tight_layout(); plt.show()
print("Interpretasi: ukuran kata pada word cloud mencerminkan dominasi visual/frekuensi relatif, bukan kekuatan prediktif. Kesimpulan utama tetap harus mengacu pada tabel frekuensi, n-gram, TF-IDF, dan statistik per kelas.")

### 5.15 Analisis Mendalam Per Label/Kelas
Ringkasan membandingkan jumlah dokumen, panjang teks, dan prevalensi elemen digital/emosional. Top words, bigrams, dan TF-IDF ditampilkan per kelas.

In [ ]:
feature_masks={name:eda_text.str.contains(pat,regex=True,na=False) for name,pat in patterns.items()}
class_rows=[]; per_class_words={}; per_class_bigrams2={}
if main_label_col:
    for lab in sorted(df[main_label_col].dropna().unique()):
        mask=df[main_label_col].eq(lab)
        toks=[]
        for s in eda_clean_text[mask]: toks.extend(re.findall(r'(?u)\b\w+\b',s))
        toks=[t for t in toks if t not in stopwords_id]
        per_class_words[str(lab)]=pd.DataFrame(Counter(toks).most_common(10),columns=['Kata','Frekuensi'])
        try: per_class_bigrams2[str(lab)]=top_ngrams(eda_clean_text[mask],(2,2),10,min_df=1)
        except ValueError: per_class_bigrams2[str(lab)]=pd.DataFrame(columns=['N-Gram','Frekuensi'])
        row={'Label':lab,'Jumlah Dokumen':int(mask.sum()),'Rata-rata Karakter':round(eda_char_count[mask].mean(),2),'Median Karakter':round(eda_char_count[mask].median(),2),'Rata-rata Kata':round(eda_word_count[mask].mean(),2),'Median Kata':round(eda_word_count[mask].median(),2)}
        for key in ['URL','Mention @username','Hashtag','Emoji (heuristik Unicode)','Huruf kapital berlebihan','Tanda baca berlebihan']:
            row[f'{key} (%)']=round(feature_masks[key][mask].mean()*100,2)
        class_rows.append(row)
    class_summary=pd.DataFrame(class_rows); display(class_summary)
    for lab in per_class_words:
        print(f"\n=== Label {lab} ===")
        print("Top words:"); display(per_class_words[lab])
        print("Top bigrams:"); display(per_class_bigrams2[lab])
        if lab in per_class_tfidf:
            print("Top TF-IDF:"); display(per_class_tfidf[lab].head(10))
print("Interpretasi: perbedaan antar kelas pada panjang teks, frasa dominan, dan elemen digital dapat menjadi sinyal tambahan, tetapi harus divalidasi pada data split yang bebas leakage sebelum dianggap prediktif.")

### 5.16 Analisis Waktu
Analisis hanya dijalankan bila dataset benar-benar memiliki kolom tanggal/waktu yang dapat diparse.

In [ ]:
time_candidates=[c for c in df.columns if any(k in c.lower() for k in ['date','time','timestamp','created_at','datetime'])]
valid_time_col=None
for c in time_candidates:
    parsed=pd.to_datetime(df[c],errors='coerce')
    if parsed.notna().mean()>=0.5:
        valid_time_col=c; eda_time=parsed; break
if valid_time_col is None:
    print("Analisis waktu tidak dilakukan karena dataset tidak memiliki informasi temporal yang relevan.")
else:
    time_df=pd.DataFrame({'time':eda_time,'label':df[main_label_col] if main_label_col else np.nan}).dropna(subset=['time'])
    time_df['period']=time_df['time'].dt.to_period('M').dt.to_timestamp()
    counts_time=time_df.groupby('period').size()
    fig,ax=plt.subplots(figsize=(12,5)); ax.plot(counts_time.index,counts_time.values,marker='o'); ax.set_title(f'Jumlah Dokumen per Bulan — {valid_time_col}'); ax.set_xlabel('Periode'); ax.set_ylabel('Jumlah Dokumen'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
    if main_label_col:
        pivot=pd.crosstab(time_df['period'],time_df['label'])
        ax=pivot.plot(figsize=(12,5)); ax.set_title('Distribusi Label per Periode'); ax.set_xlabel('Periode'); ax.set_ylabel('Jumlah Dokumen'); plt.tight_layout(); plt.show()

### 5.17 Pemeriksaan Khusus Hate Speech: Leakage, Imbalance, dan Sinyal yang Perlu Dipertahankan

In [ ]:
print("=== A. Conflicting Labels ===")
print(f"Jumlah teks identik dengan label target berbeda: {conflict_text_count:,}")
print("\n=== B. Potensi Data Leakage ===")
print(f"Duplikat teks berlebih: {n_dup_text_excess:,}; duplikat setelah normalisasi ringan: {n_norm_dup_excess:,}.")
print("Rekomendasi: deduplikasi/group split berdasarkan teks ternormalisasi dilakukan sebelum train-test split agar salinan tidak bocor ke test set.")

print("\n=== C. Class Imbalance ===")
if main_label_col:
    bc=df.loc[df[main_label_col].isin([0,1]),main_label_col].value_counts()
    if len(bc)>=2:
        ratio=float(bc.max()/bc.min()); print(f"Rasio mayoritas:minoritas = {ratio:.2f}:1")
        print("Evaluasi yang disarankan: precision, recall, F1-score, macro F1, confusion matrix. Jika imbalance signifikan, pertimbangkan class_weight/focal loss/resampling yang sesuai setelah split.")

print("\n=== D. Informasi yang Tidak Boleh Dihapus Sembarangan ===")
important_signals=[]
for k in ['Hashtag','Emoji (heuristik Unicode)','Huruf kapital berlebihan','Tanda baca berlebihan']:
    v=int(feature_masks[k].sum()); important_signals.append((k,v,round(v/len(df)*100,2)))
for n in sorted(negation_keep):
    if all_counter[n]>0: important_signals.append((f'Negasi: {n}',all_counter[n],None))
display(pd.DataFrame(important_signals,columns=['Sinyal','Jumlah/Frekuensi','Persentase Dokumen (jika relevan)']))
print("Slang, kata kasar/offensive terms, negasi, emoji, hashtag, kapital, dan tanda baca berlebihan berpotensi membawa intensitas, target, stance, atau polaritas. Karena itu pembersihan harus selektif dan berbasis bukti EDA.")

---
# Kesimpulan Exploratory Data Analysis
Tabel berikut dibentuk dari variabel hasil EDA sehingga angka tidak ditulis manual.

In [ ]:
# Menyusun temuan aktual secara programatik.
findings=[]
def addf(temuan,bukti,dampak,rekom): findings.append({'Temuan EDA':temuan,'Bukti/Hasil':bukti,'Dampak':dampak,'Rekomendasi':rekom})
addf('Ukuran dataset',f'{len(df):,} baris × {df.shape[1]} kolom','Menentukan skala analisis/pemodelan','Gunakan pipeline yang efisien dan reproducible')
addf('Missing value',f'{int(df.isna().sum().sum()):,} nilai NaN total; teks tidak valid {int(invalid_text.sum()):,}','Nilai kosong dapat mengganggu parsing/tokenisasi','Tangani hanya kolom/baris yang memang bermasalah')
addf('Duplikasi',f'{n_dup_text_excess:,} duplikat teks berlebih; {n_norm_dup_excess:,} setelah normalisasi ringan','Bias frekuensi dan potensi leakage','Review lalu deduplikasi/grouping sebelum split')
addf('Conflicting labels',f'{conflict_text_count:,} teks unik dengan label target berbeda','Label noise/ambiguity','Review konflik; gunakan aturan konsensus yang konsisten')
if main_label_col:
    bc=df.loc[df[main_label_col].isin([0,1]),main_label_col].value_counts()
    ratio_txt=f'{bc.max()/bc.min():.2f}:1' if len(bc)>=2 and bc.min()>0 else 'tidak dapat dihitung'
    addf('Distribusi kelas',label_distribution.to_dict('records') if label_distribution is not None else 'N/A','Kelas minoritas dapat kurang terpelajari',f'Pantau macro F1; rasio mayoritas:minoritas {ratio_txt}')
addf('Panjang teks',f'Median {eda_word_count.median():.0f} kata; Q1 {eda_word_count.quantile(.25):.0f}; Q3 {eda_word_count.quantile(.75):.0f}; max {eda_word_count.max():.0f}','Ada variasi panjang dan truncation risk','Tentukan max_length menggunakan distribusi token tokenizer Transformer')
addf('Vocabulary',f'{unique_tokens:,} unique dari {total_tokens:,} token; lexical diversity {lex_div:.4f}','Vocabulary besar dapat meningkatkan sparsity pada model klasik','Gunakan tokenisasi/subword; jangan normalisasi agresif tanpa validasi')
addf('Rare words',f'{len(rare1):,} token muncul sekali; {len(rare5):,} token muncul ≤5 kali','Bisa berisi typo, nama, slang, atau istilah hate speech penting','Jangan hapus otomatis; klasifikasikan/normalisasi selektif')
addf('Kata/frasa dominan',f"Top kata sesudah stopword: {', '.join(top_after['Kata'].head(5))}; top bigram: {', '.join(bi_top['N-Gram'].head(5))}",'Menunjukkan konteks linguistik corpus','Gunakan bersama TF-IDF/per-label analysis, bukan sebagai bukti tunggal')
addf('Karakter khusus', '; '.join(f"{r['Elemen']}={r['Persentase']:.2f}%" for _,r in element_table.iterrows()),'Elemen digital dapat membawa sinyal emosional/kontekstual','Normalisasi selektif; pertahankan sinyal penting')
addf('Slang/typo',f'{len(slang_df)} kandidat slang/singkatan yang terdeteksi; {len(elong_df)} kandidat elongasi ditampilkan','Meningkatkan variasi vocabulary','Bangun kamus normalisasi dari bentuk yang benar-benar muncul')
addf('Potensi leakage',f'{n_norm_dup_excess:,} normalized duplicates','Evaluasi test dapat terlalu optimistis','Deduplikasi/group split sebelum train-test split')
addf('Kebutuhan preprocessing','Berdasarkan tabel elemen, slang, duplikasi, panjang teks, dan stopword di atas','Preprocessing yang salah dapat membuang sinyal hate speech','Terapkan hanya langkah yang didukung hasil EDA')
eda_conclusion=pd.DataFrame(findings)
display(eda_conclusion)

# Rekomendasi Preprocessing Berdasarkan EDA
Keputusan berikut dibuat dari statistik yang sudah dihitung. Status dapat berubah bila data sumber berubah dan notebook dijalankan ulang.

In [ ]:
preproc=[]
def rec(step,status,reason): preproc.append({'Langkah':step,'Rekomendasi':status,'Alasan berbasis EDA':reason})
rec('Lowercase','Pertimbangkan, bukan otomatis',f"ALLCAPS terdeteksi pada {feature_masks['Huruf kapital berlebihan'].mean()*100:.2f}% dokumen; jika lowercase, pertahankan penanda intensitas ALLCAPS.")
rec('URL removal/replacement','Ganti token <URL>' if feature_masks['URL'].any() else 'Tidak prioritas',f"URL muncul pada {feature_masks['URL'].sum():,} dokumen ({feature_masks['URL'].mean()*100:.2f}%).")
rec('Mention removal/replacement','Ganti token <USER>' if feature_masks['Mention @username'].any() else 'Tidak prioritas',f"Mention muncul pada {feature_masks['Mention @username'].sum():,} dokumen ({feature_masks['Mention @username'].mean()*100:.2f}%).")
rec('Hashtag processing','Pertahankan isi; normalisasi simbol # bila perlu',f"Hashtag muncul pada {feature_masks['Hashtag'].sum():,} dokumen ({feature_masks['Hashtag'].mean()*100:.2f}%).")
rec('Emoji conversion','Pertahankan/konversi ke token deskriptif',f"Emoji terdeteksi pada {feature_masks['Emoji (heuristik Unicode)'].sum():,} dokumen ({feature_masks['Emoji (heuristik Unicode)'].mean()*100:.2f}%).")
rec('Punctuation processing','Normalisasi repetisi, jangan hapus seluruh sinyal',f"Tanda baca berlebihan muncul pada {feature_masks['Tanda baca berlebihan'].sum():,} dokumen.")
rec('Whitespace normalization','Ya',f"Newline muncul pada {feature_masks['Newline'].sum():,} dokumen; whitespace dapat diseragamkan tanpa mengubah leksikon.")
rec('Slang normalization','Ya, selektif' if len(slang_df) else 'Belum terbukti perlu',f"{len(slang_df)} bentuk slang/singkatan dari kamus eksploratif benar-benar ditemukan.")
rec('Typo/elongation normalization','Selektif',f"{len(elong_df)} kandidat elongasi teratas ditampilkan; repeated letters dapat menandai intensitas.")
rec('Stopword removal','Opsional dan konservatif',f"Stopword konservatif mencakup {prop_stop:.2f}% token; negasi dipertahankan.")
rec('Stemming','Uji, jangan otomatis','Transformer subword sering tidak membutuhkan stemming; stemming agresif dapat merusak slang/nama/kata ofensif.')
neg_found=sum(all_counter[n] for n in negation_keep)
rec('Handling negation','Wajib dipertahankan',f"Total kemunculan token negasi yang dipantau: {neg_found:,}; negasi dapat membalik makna.")
rec('Deduplication','Ya sebelum split' if n_norm_dup_excess else 'Tetap cek sebelum split',f"{n_norm_dup_excess:,} duplikat setelah normalisasi ringan; conflicting labels {conflict_text_count:,} teks.")
if main_label_col and len(bc)>=2:
    ratio=float(bc.max()/bc.min()); rec('Class imbalance','Gunakan class_weight/focal loss/resampling bila diperlukan',f"Rasio mayoritas:minoritas aktual {ratio:.2f}:1; evaluasi wajib macro F1/precision/recall/confusion matrix.")
rec('Pembatasan panjang input','Tentukan dari tokenizer model',f"Distribusi kata: P95={p95_words:.0f}, P99={p99_words:.0f}, max={eda_word_count.max():.0f}; ukur ulang dalam subword token XLM-RoBERTa.")
preprocessing_recommendations=pd.DataFrame(preproc)
display(preprocessing_recommendations)
print("Kesimpulan akhir: preprocessing tidak dilakukan permanen di notebook EDA ini. Rekomendasi di atas adalah kandidat untuk tahap data engineering setelah ditinjau berdasarkan output aktual.")

---
## Bab 6: Ekspor Data Interim

In [ ]:
# ============================================================
# KODE EKSPOR DATA INTERIM
# Uncomment (hapus tanda #) ketika siap melanjutkan ke
# tahap Data Engineering / Cleaning.
# ============================================================

# import os
# 
# # Pilih kolom yang relevan untuk disimpan
# export_cols = [
#     'text_id', 'text', 'initial_paragraph', 'topic',
#     'toxicity_label', 'identity_attack_label',
#     'threat_incitement_to_violence_label', 'insults_label',
#     'profanity_obscenity_label', 'sexually_explicit_label',
#     'polarized_label', 'related_to_election_2024_label',
#     'is_noise_or_spam_text_label',
#     'char_count', 'word_count', 'num_annotators'
# ]
# 
# df_export = df[export_cols].copy()
# 
# # Buat folder jika belum ada
# os.makedirs('data/interim', exist_ok=True)
# 
# # Simpan ke CSV
# df_export.to_csv('data/interim/data_parsed.csv', index=False)
# print(f"Data berhasil disimpan ke data/interim/data_parsed.csv")
# print(f"   Jumlah baris: {len(df_export):,}")
# print(f"   Jumlah kolom: {len(df_export.columns)}")